# Data Cleaning and Sales Analysis

Explore the sample sales data, apply the reusable cleaning functions, and generate the project visualizations. Run cells from top to bottom.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_cleaning import clean_sales_data, missing_value_summary
from src.visualization import (
    plot_correlation_heatmap, plot_dashboard, plot_missing_values,
    plot_outliers, plot_sales_distribution,
)

DATA_PATH = ROOT / 'data' / 'sales_data.csv'
IMAGE_DIR = ROOT / 'images'
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

## Inspect the raw data

Review the schema and missing-value counts before making changes. The missingness plot is based on the original input.

In [ ]:
raw = pd.read_csv(DATA_PATH)
print(f'Raw rows: {len(raw)}')
display(raw.head())
display(raw.dtypes.to_frame('dtype'))
display(missing_value_summary(raw).to_frame('missing_count'))
plot_missing_values(raw, IMAGE_DIR)

## Clean and validate

Duplicate order IDs are removed, labels and types are normalized, missing values are imputed, and sales are calculated. IQR outliers are retained and marked for review.

In [ ]:
sales = clean_sales_data(raw)
sales.to_csv(DATA_PATH.with_name('cleaned_sales_data.csv'), index=False)
print(f'Clean rows: {len(sales)}')
print(f'Flagged sales outliers: {int(sales["is_sales_outlier"].sum())}')
display(sales.head())
display(sales.groupby('region', observed=True)['sales'].agg(['count', 'sum', 'mean']).round(2))
display(sales.groupby('product', observed=True)['sales'].sum().sort_values(ascending=False).to_frame('sales'))

## Visualize the cleaned data

The figures are saved in the repository's `images/` directory. The dashboard combines regional, product, order-value, and monthly views.

In [ ]:
figure_paths = [
    plot_outliers(sales, IMAGE_DIR),
    plot_correlation_heatmap(sales, IMAGE_DIR),
    plot_sales_distribution(sales, IMAGE_DIR),
    plot_dashboard(sales, IMAGE_DIR),
]
[str(path.relative_to(ROOT)) for path in figure_paths]